# Chapter 9 — Actions and angles

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vasilybelokurov/galactic-dynamics-tutorial/blob/main/notebooks/09_actions_angles.ipynb)

**Goal.** We have met actions and angles several times as outputs of `agama.ActionFinder` (Chapters 3, 5, 6).
This chapter explains what they are, computes them by hand where we can, tests AGAMA's methods against
exact results, and shows why actions are the natural labels for orbits in a galaxy.

**You will learn**
1. the action of a one-dimensional oscillation as a phase-space area, and its **adiabatic invariance**;
2. the three actions $(J_r, J_z, J_\phi)$ of a spherical potential, computed by quadrature and compared with the
   exact isochrone result and AGAMA;
3. that frequencies are derivatives of the energy with respect to the actions;
4. how AGAMA estimates actions in a non-spherical potential (the **Stäckel fudge**), how accurate that is, and
   the inverse map from actions and angles to positions and velocities (**torus mapping**).

**Units.** $M_\odot$, kpc, km/s; actions in kpc km/s. §9.1 uses dimensionless units.

In [ ]:
import sys, importlib.util
if importlib.util.find_spec('agama') is None:      # install AGAMA on Colab (see Chapter 0)
    v = f'cp{sys.version_info.major}{sys.version_info.minor}'
    !pip install -q "https://github.com/vasilybelokurov/galactic-dynamics-tutorial/releases/download/agama-wheels/agama-1.0-{v}-{v}-linux_x86_64.whl"
import agama

agama.setUnits(mass=1, length=1, velocity=1)       # Msun, kpc, km/s -- before creating any AGAMA object
G = agama.G

import os
import numpy as np
import matplotlib.pyplot as plt
from scipy.integrate import quad
from scipy.optimize import brentq

rng = np.random.default_rng(9)

## 9.1 The action of a 1-d oscillation, and why it is special

A star oscillating in one dimension traces a closed curve in the $(x, v)$ plane. Its **action** is the area
enclosed by that curve divided by $2\pi$:

$$ J = \frac{1}{2\pi}\oint v\, dx . $$

For the harmonic oscillator, $E = \tfrac12 v^2 + \tfrac12\omega^2 x^2$, the curve is an ellipse with semi-axes
$\sqrt{2E}/\omega$ and $\sqrt{2E}$, so $J = E/\omega$. The **angle** $\theta$ is the phase of the oscillation, which grows
uniformly: $\theta = \omega t + {\rm const}$. In the variables $(\theta, J)$ the motion is trivial: $J$ stays fixed and $\theta$
increases at the rate $\omega = dE/dJ$.

Why prefer $J$ to $E$? Galaxies change: they accrete mass, and disks grow. If the potential changes **slowly**
compared with the orbital period, the energy of a star changes, but its action does not: the action is an
**adiabatic invariant** (Binney & Tremaine §3.6). Test it: let $\omega$ grow from 1 to 2 over a time $T_{\rm change}$,
and integrate with the leapfrog of Chapter 2.

In [ ]:
def oscillate(T_change, w0=1.0, w1=2.0, dt=0.002):
    """Harmonic oscillator whose frequency grows linearly from w0 to w1 during 0 < t < T_change.
    Starts at x = 1, v = 0 (J = 0.5) and runs until 20 time units after the change.
    Returns the times, E(t) and J(t) = E/omega."""
    omega = lambda t: w0 + (w1 - w0) * min(t / T_change, 1.0)
    x, v, t = 1.0, 0.0, 0.0
    out = []
    while t < T_change + 20:
        v -= 0.5 * dt * omega(t)**2 * x             # leapfrog: kick - drift - kick (Chapter 2)
        x += dt * v
        t += dt
        v -= 0.5 * dt * omega(t)**2 * x
        E = 0.5 * v**2 + 0.5 * omega(t)**2 * x**2
        out.append((t, E, E / omega(t)))
    return np.array(out).T

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
J_final = {}
for T_change in [1, 10, 100, 1000]:
    t, E, J = oscillate(T_change)
    J_final[T_change] = J[-1]
    ax[0].plot(t / T_change, E, label=f'$T_{{\\rm change}}$ = {T_change}')
    ax[1].plot(t / T_change, J)
    print(f'T_change = {T_change:5d} ({T_change / (2 * np.pi):6.1f} initial periods): '
          f'E: 0.5 -> {E[-1]:.4f},  J: 0.5 -> {J[-1]:.5f}')
for a in ax:
    a.set_xlim(0, 1.5); a.set_xlabel('$t / T_{\\rm change}$')
ax[0].set_ylabel('$E$'); ax[1].set_ylabel('$J = E/\\omega$'); ax[0].legend()
plt.tight_layout(); plt.show()
assert abs(J_final[1000] / 0.5 - 1) < 1e-3 and abs(J_final[1] / 0.5 - 1) > 0.1

When $\omega$ changes slowly (hundreds of periods) the energy doubles, as $\omega$ does, while the action stays at 0.5 to
within $10^{-4}$. A sudden change (shorter than a period) changes the action by tens of per cent, by an amount
that depends on the phase of the oscillation at the time. This is why actions are used to label the stars of a
galaxy that has grown slowly: they remember the orbits the stars were born on.

## 9.2 Actions in a spherical potential

In a spherical potential an orbit has three independent oscillations (Chapter 3): in radius, in latitude
above its orbital plane, and in azimuth. Each has an action:

- **radial action**: $J_r = \dfrac{1}{\pi}\displaystyle\int_{r_p}^{r_a} \sqrt{2[E - \Phi(r)] - L^2/r^2}\; dr$, the area of the orbit in the $(r, v_r)$ plane over $2\pi$;
- **vertical action**: $J_z = L - |L_z|$ — zero for an orbit in the equatorial plane, $L$ for a polar orbit;
- **azimuthal action**: $J_\phi = L_z$.

$J_r = 0$ for a circular orbit and grows with eccentricity. For the **isochrone** potential (Chapter 3) the integral
can be done analytically:

$$ J_r = \frac{GM}{\sqrt{-2E}} - \frac12\left(L + \sqrt{L^2 + 4GMb}\right). $$

We compute $J_r$ by quadrature (pericentre and apocentre by `brentq`, as in Chapter 1), from the exact formula,
and with AGAMA, for random phase-space points.

In [ ]:
M, b = 1e11, 3.0
iso = agama.Potential(type='Isochrone', mass=M, scaleRadius=b)
Phi_iso = lambda r: -G * M / (b + np.sqrt(b**2 + r**2))

def Jr_quadrature(E, L, Phi):
    """Radial action by direct integration of v_r between pericentre and apocentre."""
    vr2 = lambda r: 2 * (E - Phi(r)) - L**2 / r**2          # v_r^2 as a function of radius
    r_try = np.geomspace(1e-3, 1e3, 4000)
    crossing = np.where(np.diff(np.sign(vr2(r_try))) != 0)[0]   # where v_r^2 changes sign
    r_peri = brentq(vr2, r_try[crossing[0]], r_try[crossing[0] + 1])
    r_apo = brentq(vr2, r_try[crossing[-1]], r_try[crossing[-1] + 1])
    return quad(lambda r: np.sqrt(max(vr2(r), 0)), r_peri, r_apo, limit=200)[0] / np.pi

Jr_isochrone = lambda E, L: G * M / np.sqrt(-2 * E) - 0.5 * (L + np.sqrt(L**2 + 4 * G * M * b))

xv = np.column_stack([rng.uniform(-10, 10, (300, 3)), rng.normal(0, 120, (300, 3))])
E = iso.potential(xv[:, :3]) + 0.5 * np.sum(xv[:, 3:]**2, axis=1)
xv, E = xv[E < 0], E[E < 0]                          # keep bound orbits only
Lvec = np.cross(xv[:, :3], xv[:, 3:])                # angular momentum vector r x v for each point
L, Lz = np.linalg.norm(Lvec, axis=1), Lvec[:, 2]

J_agama = agama.ActionFinder(iso)(xv)                # columns: J_r, J_z, J_phi  (note the order!)
J_quad = np.array([Jr_quadrature(e, l, Phi_iso) for e, l in zip(E, L)])
J_exact = Jr_isochrone(E, L)

fig, ax = plt.subplots(figsize=(6.5, 4.5))
ax.loglog(J_exact, J_quad, 'o', ms=5, mfc='none', label='quadrature')
ax.loglog(J_exact, J_agama[:, 0], '.', ms=4, label='AGAMA')
ax.plot([1, 3e4], [1, 3e4], 'k-', lw=0.5)
ax.set_xlabel('exact $J_r$ [kpc km/s]'); ax.set_ylabel('computed $J_r$'); ax.legend(); plt.show()

print(f'{len(E)} bound orbits; max relative error of J_r: quadrature {np.max(np.abs(J_quad / J_exact - 1)):.1e}, '
      f'AGAMA {np.max(np.abs(J_agama[:, 0] / J_exact - 1)):.1e}')
print(f'max |J_z - (L - |L_z|)| = {np.max(np.abs(J_agama[:, 1] - (L - np.abs(Lz)))):.1e},  '
      f'max |J_phi - L_z| = {np.max(np.abs(J_agama[:, 2] - Lz)):.1e} kpc km/s')
assert np.max(np.abs(J_quad / J_exact - 1)) < 1e-6 and np.max(np.abs(J_agama[:, 0] / J_exact - 1)) < 1e-6
assert np.allclose(J_agama[:, 1], L - np.abs(Lz), atol=1e-6 * L.max()) and np.allclose(J_agama[:, 2], Lz)

All three agree. For a spherical potential AGAMA's `ActionFinder` does exactly this one-dimensional integral,
so it is exact up to round-off.

## 9.3 Frequencies are derivatives of the energy

Because the actions are conserved, the energy of an orbit can be written as a function of them alone, $H(\mathbf J)$.
Hamilton's equations in angle–action variables then say

$$ \dot{\mathbf J} = 0, \qquad \dot\theta_i = \Omega_i = \frac{\partial H}{\partial J_i} . $$

For the isochrone, inverting the formula above gives $H(\mathbf J)$ explicitly:

$$ H = -\frac{(GM)^2}{2\left[J_r + \tfrac12\left(L + \sqrt{L^2 + 4GMb}\right)\right]^2}, \qquad L = J_z + |J_\phi| , $$

from which $\Omega_r = \partial H/\partial J_r = (GM)^2/[J_r + \tfrac12(L + \sqrt{L^2+4GMb})]^3$. We check this, and the
general relation $\Omega_r = 1/(\partial J_r/\partial E)_L$ using finite differences of our quadrature, against
AGAMA's frequencies. (Chapter 3 found $\Omega_r$ from the radial period; this is the same quantity.)

In [ ]:
xv5 = xv[:5]
_, _, Om_agama = agama.ActionFinder(iso)(xv5, angles=True)    # actions, angles, frequencies
E5, L5 = E[:5], L[:5]
Om_formula = (G * M)**2 / (Jr_isochrone(E5, L5) + 0.5 * (L5 + np.sqrt(L5**2 + 4 * G * M * b)))**3
dE = 1e-4 * np.abs(E5)
Om_numeric = 2 * dE / np.array([Jr_quadrature(e + de, l, Phi_iso) - Jr_quadrature(e - de, l, Phi_iso)
                                for e, de, l in zip(E5, dE, L5)])
for i in range(5):
    print(f'Omega_r: AGAMA {Om_agama[i, 0]:8.3f}, dH/dJ_r formula {Om_formula[i]:8.3f}, '
          f'1/(dJ_r/dE) numerical {Om_numeric[i]:8.3f}  km/s/kpc')
assert np.allclose(Om_agama[:, 0], Om_formula, rtol=1e-6) and np.allclose(Om_numeric, Om_formula, rtol=1e-5)

## 9.4 Actions in a flattened galaxy: the Stäckel fudge and tori

In an axisymmetric potential $J_\phi = L_z$ is still exact, but $J_r$ and $J_z$ require a third integral of motion, which
in general has no formula (Chapter 5). There is one family of potentials where it does: **Stäckel potentials**,
which separate in prolate spheroidal coordinates; their actions are again one-dimensional integrals.

The **Stäckel fudge** (Binney 2012) pretends that the real potential is, locally, of Stäckel form, and evaluates
those integrals using the real potential. It is the default method of `ActionFinder` for axisymmetric potentials
(the coordinate system's focal distance is chosen automatically for each orbit). Since it is an approximation,
the "actions" it returns are not exactly constant along a real orbit. We measure how much they vary for orbits of
increasing size in MWPotential2014.

In [ ]:
mw = agama.Potential(os.path.join(os.path.dirname(agama.__file__), 'data', 'MWPotential2014.ini'))
af_mw = agama.ActionFinder(mw)
R0 = 8.0
vc0 = np.sqrt(-R0 * mw.force([R0, 0, 0])[0])

fig, ax = plt.subplots(1, 2, figsize=(13, 4.5))
rows = []
for j, v_kick in enumerate([10, 30, 60, 100, 150]):  # equal extra radial and vertical velocity, km/s
    t, traj = agama.orbit(potential=mw, ic=[R0, 0, 0, v_kick, vc0, v_kick], time=5.0, trajsize=2001)
    J = af_mw(traj)                                   # Staeckel-fudge actions at every point of the orbit
    ax[0].plot(t, J[:, 0] / J[:, 0].mean() - 1, lw=0.7, label=f'$v_R = v_z$ = {v_kick} km/s')
    ax[1].plot(t, J[:, 1] / J[:, 1].mean() - 1, lw=0.7)
    rows.append((v_kick, J[:, 0].mean(), np.std(J[:, 0]) / J[:, 0].mean(), J[:, 1].mean(), np.std(J[:, 1]) / J[:, 1].mean()))
ax[0].set_ylabel('$J_r$ / mean $- 1$'); ax[1].set_ylabel('$J_z$ / mean $- 1$')
for a in ax:
    a.set_xlabel('$t$ [AGAMA time units, 0.98 Gyr]')
ax[0].legend(fontsize=8); plt.tight_layout(); plt.show()

for v_kick, Jr_m, Jr_s, Jz_m, Jz_s in rows:
    print(f'v_R = v_z = {v_kick:3d} km/s: J_r = {Jr_m:6.1f} (rms variation {Jr_s:.1e}), J_z = {Jz_m:6.1f} ({Jz_s:.1e})')
assert max(max(r[2], r[4]) for r in rows) < 0.01

The fudge holds the actions constant to 0.01–0.6% rms (peak-to-peak excursions reach 1.5%): excellent for disk
stars, worse for hot orbits that reach far above the plane. For the two hottest orbits $J_z$ jumps between two
levels: an artefact of the approximation, not of the orbit (Exercise 3 explores the role of the focal distance).
Compare Chapter 5, where the vertical energy of a modest orbit varied by 19%.

**The inverse map.** Given $(\mathbf J, \boldsymbol\theta)$, where is the star? Sweeping the three angles at fixed actions traces
out the whole orbit — a 3-d **torus** in 6-d phase space. AGAMA's `ActionMapper` constructs such tori (the
"Torus Machine", McGill & Binney 1990; Binney & McMillan 2016). Take one orbit's mean actions, draw 4000 random
angles, map them to positions, and overlay the integrated orbit in the meridional plane.

In [ ]:
t, traj = agama.orbit(potential=mw, ic=[R0, 0, 0, 60, vc0, 60], time=5.0, trajsize=4001)
J_orbit = af_mw(traj).mean(axis=0)                   # (J_r, J_z, J_phi) averaged along the orbit

am = agama.ActionMapper(mw)
angles = rng.uniform(0, 2 * np.pi, (4000, 3))
xv_torus = am(np.column_stack([np.tile(J_orbit, (4000, 1)), angles]))   # input rows: J_r, J_z, J_phi, theta_r, theta_z, theta_phi

fig, ax = plt.subplots(figsize=(7, 4.5))
ax.plot(np.hypot(traj[:, 0], traj[:, 1]), traj[:, 2], '-', lw=0.3, color='gray', label='integrated orbit')
ax.plot(np.hypot(xv_torus[:, 0], xv_torus[:, 1]), xv_torus[:, 2], '.', ms=2, color='C1', label='torus points')
ax.set_xlabel('$R$ [kpc]'); ax.set_ylabel('$z$ [kpc]'); ax.legend(); plt.show()

J_back = af_mw(xv_torus)                             # Staeckel actions of the torus points
print(f'orbit actions (J_r, J_z, J_phi): {np.round(J_orbit, 2)}')
print(f'actions recovered from torus points: mean {np.round(J_back.mean(axis=0), 2)}, '
      f'rms relative scatter J_r {np.std(J_back[:, 0]) / J_orbit[0]:.1e}, J_z {np.std(J_back[:, 1]) / J_orbit[1]:.1e}')
assert np.allclose(J_back.mean(axis=0), J_orbit, rtol=1e-3)

The torus and the integrated orbit fill the same region. The Stäckel actions of the torus points scatter by
0.3–0.5% around the input values, and their mean matches the orbit's actions to 0.1% or better. The two methods are independent approximations — one finds actions from positions, the
other positions from actions — so their agreement measures the accuracy of both.

Why bother? Because in action space the dynamics is trivial, an equilibrium galaxy can be described by a DF
$f(\mathbf J)$, which by the Jeans theorem (Chapter 6) is automatically steady. That is the subject of Chapter 10.

## Summary

- An action is a phase-space area over $2\pi$; the conjugate angle grows uniformly at the frequency $\Omega = \partial H/\partial J$.
- Actions are adiabatic invariants: conserved when the potential changes slowly, while the energy is not.
- Spherical potentials: $J_r$ is a 1-d integral, $J_z = L - |L_z|$, $J_\phi = L_z$; exact in AGAMA.
- Axisymmetric potentials: the Stäckel fudge estimates $J_r$, $J_z$ to better than 1% for disk orbits; tori give the
  inverse map.

| task | AGAMA |
|---|---|
| actions (order $J_r, J_z, J_\phi$) | `agama.ActionFinder(pot)(xv)` |
| actions, angles, frequencies | `J, theta, Omega = agama.ActionFinder(pot)(xv, angles=True)` |
| Stäckel actions with a chosen focal distance | `agama.actions(pot, xv, fd=...)` |
| positions and velocities from actions and angles | `agama.ActionMapper(pot)(np.column_stack([J, theta]))` |

## Exercises

1. **Adiabatic invariance, sudden limit.** For $T_{\rm change} = 1$ in §9.1, repeat with different starting phases (start at
   $x = \cos\phi_0$, $v = -\sin\phi_0$). Plot the final $J$ against $\phi_0$.
2. **Plummer actions.** Compute $J_r$ by quadrature for the Plummer potential (no closed formula) and compare with AGAMA.
3. **Focal distance.** Use `agama.actions(mw, traj, fd=...)` with several fixed focal distances (0.5, 2, 5 kpc) for one of the
   orbits of §9.4. Which value keeps the actions most nearly constant?
4. **Frequencies from tori.** `ActionMapper` returns frequencies with `frequencies=True`. Compare them with `ActionFinder`
   frequencies at the torus points.

## References

- J. Binney & S. Tremaine, *Galactic Dynamics*, 2nd ed. (Princeton, 2008), §3.5 and §3.6.
- J. Bovy, *Dynamics and Astrophysics of Galaxies*, chapter 4 and §9: https://galaxiesbook.org
- J. Binney (2012), "Actions for axisymmetric potentials", MNRAS 426, 1324: https://ui.adsabs.harvard.edu/abs/2012MNRAS.426.1324B
- C. McGill & J. Binney (1990), "Torus construction in general gravitational potentials", MNRAS 244, 634:
  https://ui.adsabs.harvard.edu/abs/1990MNRAS.244..634M
- J. Binney & P. J. McMillan (2016), "Torus mapper", MNRAS 456, 1982: https://ui.adsabs.harvard.edu/abs/2016MNRAS.456.1982B
- J. L. Sanders & J. Binney (2016), "A review of action estimation methods for galactic dynamics", MNRAS 457, 2107:
  https://ui.adsabs.harvard.edu/abs/2016MNRAS.457.2107S
- E. Vasiliev (2019), AGAMA, MNRAS 482, 1525: https://doi.org/10.1093/mnras/sty2672